# Лабораторная работа №5

## Дерево решений

Классификация деревом по критерию Джини. Узел делится по одному признаку и порогу, качество разбиения измеряется приростом Джини. Построение останавливается, если прироста нет или достигнута максимальная глубина. Разбиение, после которого в листе оказалось бы меньше `min_samples_leaf` объектов, не рассматривается.

В конце ноутбука берётся то же разбиение, что в ЛР 4 (`../task4/data.csv`, травма в следующем сезоне), и самописное дерево сравнивается с `DecisionTreeClassifier` при параметрах лучшего CART из четвёртой работы: `criterion="gini"`, `max_depth=6`, `min_samples_leaf=10`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

## Узел и лист

Классы из шаблона задания. Лист запоминает объекты, которые в него попали, и в качестве прогноза берёт класс, встретившийся чаще всего.

In [2]:
class Node:

    def __init__(self, index, t, true_branch, false_branch):
        self.index = index  # индекс признака, по которому ведется сравнение с порогом в этом узле
        self.t = t  # значение порога
        self.true_branch = true_branch  # поддерево, удовлетворяющее условию в узле
        self.false_branch = false_branch  # поддерево, не удовлетворяющее условию в узле


class Leaf:

    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
        self.prediction = self.predict()

    def predict(self):
        # подсчет количества объектов разных классов
        classes = {}  # сформируем словарь "класс: количество объектов"
        for label in self.labels:
            if label not in classes:
                classes[label] = 0
            classes[label] += 1

        # найдем класс, количество объектов которого будет максимальным в этом листе и вернем его
        prediction = max(classes, key=classes.get)
        return prediction

## Критерий Джини

`labels` — метки объектов в узле, одномерный массив из 0 и 1. Доля класса 1 равна сумме меток, делённой на число объектов. Критерий равен `1 - ((1 - p)² + p²)`. У однородного узла он равен 0, при равных долях классов — 0.5.

In [3]:
def gini(labels):
    p_class = labels.sum() / labels.shape[0]
    return 1 - ((1-p_class) ** 2 + p_class ** 2)

## Прирост информации

`root_gini` — критерий Джини родительского узла до разбиения. Из него вычитается взвешенное по числу объектов среднее критериев Джини левой части (`признак <= t`) и правой (`признак > t`). Чем чище потомки относительно родителя, тем больше прирост.

In [4]:
def gain(left_labels, right_labels, root_gini):
    left_gini = (left_labels.shape[0] / (left_labels.shape[0] + right_labels.shape[0])) * gini(left_labels)
    right_gini = (right_labels.shape[0] / (left_labels.shape[0] + right_labels.shape[0])) * gini(right_labels)
    return root_gini - (left_gini + right_gini)

## Разбиение датасета в узле

Готовая функция из шаблона. Левая часть — объекты с `data[:, column_index] <= t`.

In [5]:
def split(data, labels, column_index, t):
    left = np.where(data[:, column_index] <= t)
    right = np.where(data[:, column_index] > t)

    true_data = data[left]
    false_data = data[right]

    true_labels = labels[left]
    false_labels = labels[right]

    return true_data, false_data, true_labels, false_labels

## Критерий останова: размер листа

Разбиение допускается, только если и слева, и справа не меньше `min_samples_leaf` объектов. В шаблоне задания этот порог равен 3. В сравнении с ЛР 4 передаётся 10 — размер листа у лучшего CART.

In [6]:
def split_is_allowed(true_labels, false_labels, min_samples_leaf):
    split_condition = (true_labels.shape[0] >= min_samples_leaf) & (false_labels.shape[0] >= min_samples_leaf)
    if split_condition:
        return True
    else:
        return False

## Наилучшее разбиение

Перебор признаков и порогов из шаблона. Порог — уникальное значение признака. Разбиения, которые не прошли `split_is_allowed`, пропускаются.

In [7]:
def find_best_split(data, labels, min_samples_leaf=3):
    root_gini = gini(labels)

    best_gain = 0
    best_t = None
    best_index = None

    n_features = data.shape[1]

    for index in range(n_features):
        t_values = np.unique(data[:, index])

        for t in t_values:
            true_data, false_data, true_labels, false_labels = split(data, labels, index, t)

            if not split_is_allowed(true_labels, false_labels, min_samples_leaf):
                continue

            current_gain = gain(true_labels, false_labels, root_gini)

            if current_gain > best_gain:
                best_gain, best_t, best_index = current_gain, t, index

    return best_gain, best_t, best_index

## Критерии останова при построении узла

Узел становится листом в двух случаях:

- прирост лучшего допустимого разбиения равен нулю, в том числе когда подходящего порога нет;
- достигнута максимальная глубина. У корня `depth = 0`; при `max_depth=6` узел с `depth >= 6` больше не делится. `max_depth=None` снимает ограничение по глубине.

Вместе с проверкой размера листа критериев останова три. Останов по нулевому приросту нужен и для того, чтобы не вызывать `split`, когда порог так и не найден.

In [8]:
def should_stop(gain, n_samples, depth, max_depth):
    if gain == 0:
        return True
    if max_depth is not None and depth >= max_depth:
        return True
    return False

## Построение дерева

Рекурсия из шаблона. Параметры `max_depth` и `min_samples_leaf` прокинуты в критерии останова, чтобы дерево можно было ограничить так же, как CART в ЛР 4.

In [9]:
def build_tree(data, labels, max_depth=None, min_samples_leaf=3, depth=0):
    best_gain, t, index = find_best_split(data, labels, min_samples_leaf=min_samples_leaf)

    if should_stop(best_gain, len(labels), depth, max_depth):
        return Leaf(data, labels)

    true_data, false_data, true_labels, false_labels = split(data, labels, index, t)

    true_branch = build_tree(true_data, true_labels, max_depth, min_samples_leaf, depth + 1)
    false_branch = build_tree(false_data, false_labels, max_depth, min_samples_leaf, depth + 1)

    return Node(index, t, true_branch, false_branch)

## Прогноз и печать дерева

Готовые функции из шаблона. `predict` возвращает список меток в том же порядке, что и строки `data`.

In [10]:
def classify_object(obj, node):
    # Останавливаем рекурсию, если достигли листа
    if isinstance(node, Leaf):
        answer = node.prediction
        return answer

    if obj[node.index] <= node.t:
        return classify_object(obj, node.true_branch)
    else:
        return classify_object(obj, node.false_branch)


def predict(data, tree):
    classes = []
    for obj in data:
        prediction = classify_object(obj, tree)
        classes.append(prediction)
    return classes


def print_tree(node, spacing=""):
    # Если лист, то выводим его прогноз
    if isinstance(node, Leaf):
        print(spacing + "Прогноз:", node.prediction)
        return

    # Выведем значение индекса и порога на этом узле
    print(spacing + "Индекс", str(node.index), "<=", str(node.t))

    # Рекурсионный вызов функции на положительном поддереве
    print(spacing + "--> True:")
    print_tree(node.true_branch, spacing + "  ")

    # Рекурсионный вызов функции на отрицательном поддереве
    print(spacing + "--> False:")
    print_tree(node.false_branch, spacing + "  ")

## Метрика

Доля объектов, у которых прогноз совпал с истинной меткой. `actual` и `predicted` могут быть и `list`, и `ndarray`.

In [11]:
def accuracy_metric(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    return float((actual == predicted).mean())

## Сравнение с ЛР 4

Берётся тот же датасет футболистов и та же подготовка, что в `task4/ЛР №4.ipynb`:

- целевая переменная `Injury_Next_Season`;
- `Position` кодируется one-hot с `drop_first=True`;
- выборка делится 80/20 со стратификацией и `random_state=42`.

В ЛР 4 лучший CART по сетке — `DecisionTreeClassifier(criterion="gini", max_depth=6, min_samples_leaf=10)`. На тесте у него accuracy **0.787**. Здесь то же дерево обучается ещё раз через sklearn, а рядом — самописное дерево с теми же `max_depth` и `min_samples_leaf`.

На 640 обучающих объектах перебор порогов в чистом Python занимает заметное время.

Совпадение каждого прогноза со sklearn — требование задания. Порог в этом шаблоне равен уникальному значению признака, а sklearn хранит середину между соседними значениями. На объектах обучающей выборки такие пороги обычно задают одно и то же разбиение; на тесте точка, попавшая строго между значением признака и серединой интервала, может уйти в другую ветку.

In [12]:
candidates = [
    Path("../task4/data.csv"),
    Path("task4/data.csv"),
]
data_path = next(path for path in candidates if path.exists())

df = pd.read_csv(data_path)
df = pd.get_dummies(df, columns=["Position"], prefix=["Is"], drop_first=True, dtype=int)

target = "Injury_Next_Season"
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("файл:", data_path.resolve())
print("признаков:", X.shape[1])
print("train:", X_train.shape, "доля класса 1:", round(float(y_train.mean()), 3))
print("test: ", X_test.shape, "доля класса 1:", round(float(y_test.mean()), 3))
print()
for i, name in enumerate(X.columns):
    print(f"{i:2d}  {name}")

файл: C:\Users\BSUIRHelper\Desktop\university\mlm\task4\data.csv
признаков: 20
train: (640, 20) доля класса 1: 0.5
test:  (160, 20) доля класса 1: 0.5

 0  Age
 1  Height_cm
 2  Weight_kg
 3  Training_Hours_Per_Week
 4  Matches_Played_Past_Season
 5  Previous_Injury_Count
 6  Knee_Strength_Score
 7  Hamstring_Flexibility
 8  Reaction_Time_ms
 9  Balance_Test_Score
10  Sprint_Speed_10m_s
11  Agility_Score
12  Sleep_Hours_Per_Night
13  Stress_Level_Score
14  Nutrition_Quality_Score
15  Warmup_Routine_Adherence
16  BMI
17  Is_Forward
18  Is_Goalkeeper
19  Is_Midfielder


In [13]:
# Те же ограничения, что у лучшего CART в ЛР 4.
MAX_DEPTH = 6
MIN_SAMPLES_LEAF = 10

sk_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    random_state=42,
)
sk_tree.fit(X_train, y_train)

my_tree = build_tree(
    X_train.to_numpy(),
    y_train.to_numpy(),
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
)

In [14]:
def compare_split(name, actual, sk_pred, custom_pred):
    actual = np.asarray(actual)
    sk_pred = np.asarray(sk_pred)
    custom_pred = np.asarray(custom_pred)

    match = sk_pred == custom_pred
    print(f"{name}")
    print(f"  совпало прогнозов: {int(match.sum())} из {len(match)}")
    print(f"  accuracy sklearn:  {accuracy_score(actual, sk_pred):.3f}")
    print(f"  accuracy своё:     {accuracy_metric(actual, custom_pred):.3f}")


compare_split("train", y_train, sk_tree.predict(X_train), predict(X_train.to_numpy(), my_tree))
compare_split("test", y_test, sk_tree.predict(X_test), predict(X_test.to_numpy(), my_tree))

sk_test_acc = accuracy_score(y_test, sk_tree.predict(X_test))
print()
print("CART в отчёте ЛР 4, accuracy на тесте: 0.787")
print(f"sklearn на этом же разбиении:          {sk_test_acc:.3f}")

metric_on_sklearn = accuracy_metric(y_test.to_numpy(), sk_tree.predict(X_test))
if abs(metric_on_sklearn - sk_test_acc) > 1e-9:
    raise AssertionError(
        "accuracy_metric не совпала с sklearn.metrics.accuracy_score "
        f"({metric_on_sklearn} != {sk_test_acc})"
    )
print("accuracy_metric совпадает с sklearn.metrics.accuracy_score")

train
  совпало прогнозов: 630 из 640
  accuracy sklearn:  0.905
  accuracy своё:     0.905
test
  совпало прогнозов: 160 из 160
  accuracy sklearn:  0.787
  accuracy своё:     0.787

CART в отчёте ЛР 4, accuracy на тесте: 0.787
sklearn на этом же разбиении:          0.787
accuracy_metric совпадает с sklearn.metrics.accuracy_score


Печать самописного дерева. Число в строке `Индекс` — номер столбца из ячейки с загрузкой данных.

In [15]:
print_tree(my_tree)

Индекс 13 <= 55.7188522232306
--> True:
  Индекс 9 <= 78.60272350052891
  --> True:
    Индекс 7 <= 83.11685825006852
    --> True:
      Индекс 6 <= 77.03617239096694
      --> True:
        Индекс 2 <= 76.0
        --> True:
          Прогноз: 1
        --> False:
          Прогноз: 1
      --> False:
        Прогноз: 1
    --> False:
      Прогноз: 0
  --> False:
    Индекс 8 <= 266.7009185960968
    --> True:
      Индекс 10 <= 5.5750908203406055
      --> True:
        Прогноз: 1
      --> False:
        Индекс 7 <= 71.03423972594777
        --> True:
          Прогноз: 1
        --> False:
          Индекс 12 <= 6.574896238601532
          --> True:
            Прогноз: 0
          --> False:
            Прогноз: 0
    --> False:
      Индекс 6 <= 76.07870615228917
      --> True:
        Индекс 3 <= 10.697365347398415
        --> True:
          Прогноз: 1
        --> False:
          Прогноз: 1
      --> False:
        Прогноз: 0
--> False:
  Индекс 8 <= 242.75917642543396
  --